In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

In [2]:
df = pd.read_csv('maccs_combined_df.csv')

In [3]:
df.columns

Index(['SMILES', 'PCE', '0', '1', '2', '3', '4', '5', '6', '7',
       ...
       '157', '158', '159', '160', '161', '162', '163', '164', '165', '166'],
      dtype='object', length=169)

In [4]:
columns_to_drop = df.columns[:2].tolist()
X = df.drop(columns = columns_to_drop, axis=1)

In [5]:
X.columns

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '157', '158', '159', '160', '161', '162', '163', '164', '165', '166'],
      dtype='object', length=167)

In [6]:
X.shape

(1481, 167)

In [7]:
target_df = pd.read_csv('Combined_df.csv')

In [8]:
target_df.columns

Index(['Unnamed: 0', 'SMILES', 'PCE'], dtype='object')

In [9]:
y = target_df['PCE']

In [10]:
y.shape

(1481,)

In [11]:
X.fillna(0, inplace = True)
X=X.select_dtypes(include='number')
X = X.astype(np.float32)
max_value = np.finfo(np.float32).max
X = X.loc[:, (X.iloc[:,:] <= max_value).all(axis=0)]

In [12]:
X.shape

(1481, 167)

In [13]:
def remove_low_variance(input_data, threshold=0.1):
    selection = VarianceThreshold(threshold)
    selection.fit(input_data)
    return input_data[input_data.columns[selection.get_support(indices=True)]]

In [14]:
X = remove_low_variance(X, threshold=0.1)
X

,33,41,42,50,58,62,65,66,67,75,...,134,138,140,146,148,152,153,156,157,158
0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0
1,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
1477,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0
1478,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0
1479,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0


In [15]:
X.shape

(1481, 42)

In [16]:
X.columns

Index(['33', '41', '42', '50', '58', '62', '65', '66', '67', '75', '77', '79',
       '80', '85', '87', '89', '92', '94', '98', '100', '107', '109', '110',
       '111', '112', '113', '117', '121', '122', '124', '126', '130', '134',
       '138', '140', '146', '148', '152', '153', '156', '157', '158'],
      dtype='object')

In [17]:
def correlation(dataset, threshold):
    col_corr = set()  # Set of all the names of correlated columns
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if (corr_matrix.iloc[i, j]) > threshold: # we are interested in absolute coeff value
                colname = corr_matrix.columns[i]  # getting the name of column
                col_corr.add(colname)
    return col_corr
corr_features = correlation(X, 0.8)
print('Features with correlation more than 0.8 :',len(set(corr_features)))
X = X.drop(list(corr_features & set(X.columns)), axis=1)

Features with correlation more than 0.8 : 23


In [18]:
X.shape

(1481, 19)

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
X_train.shape, X_test.shape

((1184, 19), (297, 19))

In [21]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rf = RandomForestRegressor(random_state=42)

param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)], # Number of trees in the forest
    'max_depth': [int(x) for x in np.linspace(10, 110, num = 11)], # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],                   # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],                     # Minimum number of samples required to be at a leaf node
    'max_features': ['auto', 'sqrt', 'log2'],           # Number of features to consider at every split
    'bootstrap': [True, False]                         # Whether to use bootstrap samples when building trees
}

random_search = RandomizedSearchCV(estimator=rf, 
                                   param_distributions=param_dist, 
                                   n_iter=100,        # Number of iterations for searching the parameter space
                                   scoring='r2',  # Metric to optimize
                                   cv=5,              # 5-fold cross-validation
                                   verbose=2,         # Show progress
                                   random_state=42, 
                                   n_jobs=-1)         # Use all available cores

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)

y_pred = random_search.best_estimator_.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R² Score: {r2}')

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END bootstrap=False, max_depth=70, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=600; total time=   0.9s
[CV] END bootstrap=True, max_depth=100, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=800; total time=   1.4s
[CV] END bootstrap=True, max_depth=80, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=1600; total time=   3.0s
[CV] END bootstrap=True, max_depth=90, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=1800; total time=   3.1s
[CV] END bootstrap=False, max_depth=70, max_features=log2, min_samples_leaf=4, min_samples_split=2, n_estimators=400; total time=   0.6s
[CV] END bootstrap=True, max_depth=80, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=1600; total time=   2.7s
[CV] END bootstrap=True, max_depth=70, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=2

/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
120 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
120 fits failed with the following error:
Traceback (most recent call last):
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/base.py", line 1466, in wrapper
    estimator._validate_params()
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/base.py"

Best Parameters: {'n_estimators': 1400, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 60, 'bootstrap': False}
Mean Squared Error: 6.430002771726265
R² Score: 0.6329499102910201
